In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import joblib

from sklearn_evaluation import plot

from datetime import datetime

from sklearn.ensemble import (
    AdaBoostRegressor,
    RandomForestRegressor,
    GradientBoostingRegressor,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

from pystac.client import Client
from odc.stac import load

from utils import make_indices, mask_land, mask_deeps, do_prediction

In [ ]:
MAX_DEPTH = -30.0

# Get all the csvs in the data directory
data_dir = Path("data")
regions = gpd.read_file('postcards.geojson')

# Read them and merge them into a single dataframe
all = []
atolls = []
islands = []

for region in regions.itertuples():
    csv = data_dir / f"training/{region.name}_land_mask.csv"
    is_atoll = region.type == "Atoll"

    gdf = gpd.read_file(csv)

    # Make sure everything can be converted to a float
    for col in gdf.columns:
        gdf[col] = gdf[col].astype(float)

    # Replace infinite values with NaN
    gdf = gdf.replace([float('-inf'), float('inf')], float('nan'))

    # Drop rows with missing values
    gdf = gdf.dropna()
    gdf = gdf[gdf.depth > MAX_DEPTH]

    print(f"Read {csv} ({'Atoll' if is_atoll else 'Island'}) with {len(gdf)} data points less than {MAX_DEPTH} m")

    # Get them all and put them in lists
    all.append(gdf)
    if is_atoll:
        atolls.append(gdf)
    else:
        islands.append(gdf)

all_data = gpd.GeoDataFrame(pd.concat(all, ignore_index=True))
atolls_data = gpd.GeoDataFrame(pd.concat(atolls, ignore_index=True))
islands_data = gpd.GeoDataFrame(pd.concat(islands, ignore_index=True))

print(f"\nTotal data points: {len(all_data)}, Atolls: {len(atolls_data)}, Islands: {len(islands_data)}")

In [ ]:
# Run the training using a combination of adaboost, random forest and gradient boosting
# along with all data, atolls and islands only and store the results in a dictionary, nested by data type and model

results = {}

for data, name in [(all_data, 'All'), (atolls_data, 'Atolls'), (islands_data, 'Islands')]:
    print(f"\n{name} data")
    X = data.drop(columns=['x', 'y', 'depth'])
    y = data['depth']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    for model in [AdaBoostRegressor, RandomForestRegressor, GradientBoostingRegressor]:
        model_name = model.__name__
        print(f"- {model_name}")
        reg = model()
        reg.fit(X_train, y_train)

        y_pred = reg.predict(X_test)
        mse = mean_squared_error(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)

        print(f"  - MSE: {mse}")
        print(f"  - MAE: {mae}")

        if name not in results:
            results[name] = {}
        results[name][model_name] = {
            'model': reg,
            'mse': mse,
            'mae': mae
        }

In [ ]:
# Print the results as a markdown table
print("## Results")
print("\n| Data | Model | MSE | MAE |")
print("|------|-------|-----|-----|")
for data, models in results.items():
    for model, metrics in models.items():
        print(f"| {data} | {model} | {metrics['mse']:0.3f} | {metrics['mae']:0.3f} |")


## Results

Limiting to 10 m depth, masking out land and deep water using ln(b/g) at 0.2.

| Data | Model | MSE | MAE |
|------|-------|-----|-----|
| All | AdaBoostRegressor | 1.560 | 0.878 |
| All | RandomForestRegressor | 0.574 | 0.359 |
| All | GradientBoostingRegressor | 0.747 | 0.450 |
| Atolls | AdaBoostRegressor | 3.343 | 1.780 |
| Atolls | RandomForestRegressor | 1.630 | 0.874 |
| Atolls | GradientBoostingRegressor | 1.890 | 0.995 |
| Islands | AdaBoostRegressor | 1.400 | 0.791 |
| Islands | RandomForestRegressor | 0.552 | 0.340 |
| Islands | GradientBoostingRegressor | 0.705 | 0.421 |

## Results

| Data | Model | MSE | MAE |
|------|-------|-----|-----|
| All | AdaBoostRegressor | 20.965 | 3.961 |
| All | RandomForestRegressor | 4.811 | 1.043 |
| All | GradientBoostingRegressor | 8.611 | 1.749 |
| Atolls | AdaBoostRegressor | 13.094 | 3.034 |
| Atolls | RandomForestRegressor | 5.565 | 1.592 |
| Atolls | GradientBoostingRegressor | 7.163 | 1.917 |
| Islands | AdaBoostRegressor | 18.255 | 3.650 |
| Islands | RandomForestRegressor | 4.249 | 0.924 |
| Islands | GradientBoostingRegressor | 7.966 | 1.579 |

## Results

| Data | Model | MSE | MAE |
|------|-------|-----|-----|
| All | AdaBoostRegressor | 20.96483163898461 | 3.9613885022779565 |
| All | RandomForestRegressor | 4.8105028946806625 | 1.0432409430712566 |
| All | GradientBoostingRegressor | 8.611309632799752 | 1.7492421405853138 |
| Atolls | AdaBoostRegressor | 13.09441769145558 | 3.0335415525890266 |
| Atolls | RandomForestRegressor | 5.564959169556893 | 1.5920935238286615 |
| Atolls | GradientBoostingRegressor | 7.163126672483363 | 1.9174306190555672 |
| Islands | AdaBoostRegressor | 18.25506282769581 | 3.649646061896416 |
| Islands | RandomForestRegressor | 4.249050104594601 | 0.9243737076270009 |
| Islands | GradientBoostingRegressor | 7.965919849101228 | 1.5790393716845093 |

In [ ]:
data = all_data

# Split the data into training and testing
train, test = train_test_split(data, test_size=0.3)

# Define the variables and the target
depth = train["depth"]
variables = train.drop(columns=["depth", "x", "y"])

# Define the model
# regressor = AdaBoostRegressor()
regressor = RandomForestRegressor()
# regressor = GradientBoostingRegressor()

# Train the model
model = regressor.fit(variables, depth)

# Evaluate on our test data
test_depth = test["depth"]
test_variables = test.drop(columns=["depth", "x", "y"])

predictions = model.predict(test_variables)
mse = mean_squared_error(test_depth, predictions)
mae = mean_absolute_error(test_depth, predictions)

print(f"Mean squared error: {mse:.3f}")
print(f"Mean absolute error: {mae:.3f}")

In [ ]:
from sklearn.linear_model import LinearRegression
from matplotlib import pyplot as plt
import numpy as np

from sklearn_evaluation.plot.regression import _set_ax_settings

# Do this plot, but with alpha on the points
# plot.residuals(test_depth, predictions)

y_true = test_depth
y_pred = predictions


_, ax = plt.subplots()

default_color = "#00B0FF"

# horizontal line for residual=0
ax.axhline(y=0, color=default_color)

ax.scatter(y_pred, y_true - y_pred, c=default_color, edgecolors=default_color, alpha=0.01)

_set_ax_settings(ax, "Predicted Value", "Residuals", "Residuals Plot")

In [ ]:
plot.feature_importances(model, feature_names=variables.columns)

In [ ]:
# Do this plot, but with alpha on the points
# plot.regression.prediction_error(test_depth, predictions)

_, ax = plt.subplots()
regression = LinearRegression()

if isinstance(y_true, pd.Series):
    y_true = y_true.values
y_reshaped = y_true.reshape((-1, 1))

# it is necessary to fit the model with y_true and y_pred
# to get the best fit line representing the error trend
regression.fit(y_reshaped, y_pred)
x = np.linspace(min(min(y_true), min(y_pred)), max(max(y_true), max(y_pred)))
y = regression.intercept_ + regression.coef_ * x

default_color = "#00B0FF"

ax.plot(x, y, color="#666", label="best fit", linewidth=1)

# identity line
ax.plot(
    x, x, label="identity", color="#000", linewidth=1, alpha=0.5, linestyle="dashed"
)

# scatter plot
ax.scatter(y_true, y_pred, c=default_color, edgecolors=default_color, alpha=0.01)

# R2
r2 = regression.score(y_reshaped, y_pred)
plt.plot([], [], " ", label=f"R2 = {round(r2, 5)}")

_set_ax_settings(ax, "y_true", "y_pred", "Prediction Error")
ax.legend(loc="upper left")

In [ ]:
out = "models/2025_03_11_randomforest_land_mask_30m.joblib"

# Write out the model
joblib.dump(model, out)

# Write out a little metadata file
with open(out.replace(".joblib", ".txt"), "w") as f:
    f.write(f"Mean squared error: {mse:.3f}\n")
    f.write(f"Mean absolute error: {mae:.3f}\n")
    f.write(f"Model: {regressor.__class__.__name__}\n")
    f.write(f"Data: all, limit at {MAX_DEPTH} m\n")
    f.write(f"Masked: land only\n")
    f.write(f"Date and time: {datetime.now()}\n")
    f.write(f"File: {out}\n")

In [ ]:
# Bounding box for Tuvalu
# bbox = [179.020, -8.665, 179.218, -8.413]

# Bounding box for Suva
bbox = [178.400, -18.200, 178.600, -18.000]

# Test the model on a region we haven't trained on
catalog = Client.open("https://stac.digitalearthpacific.org")
collection = "dep_s2_geomad"

items = catalog.search(collections=[collection], bbox=bbox, datetime="2024").item_collection()
print(f"Found {len(items)} items")

geomad = load(items, bbox=bbox, chunks={})
geomad = make_indices(geomad).compute()

do_prediction(geomad, model)
# depth["elevation_masked"] = mask_with_gebco(depth, -1000, interpolate=False).elevation

depth = mask_land(geomad, ds_to_mask=depth)
depth = mask_deeps(geomad, ds_to_mask=depth, threshold=1.575)

depth

In [ ]:
depth = mask_land(geomad, ds_to_mask=depth)
depth = mask_deeps(geomad, ds_to_mask=depth)

depth

In [ ]:
depth.elevation.odc.explore(cmap="Blues_r", robust=True)

In [ ]:
import folium
from ipyleaflet import basemaps

depth_masked = depth.where(~land_mask)

centroid = depth_masked.odc.geobox.geographic_extent.centroid.to_crs("EPSG:4326").coords[0]
m = folium.Map(location=[centroid[1], centroid[0]], zoom_start=12, tiles=basemaps.Esri.WorldImagery)

options = {
    "min": -40,
    "max": 0,
    "cmap": "Blues_r"
}
depth_masked.elevation.odc.add_to(m, **options, name="depth")
depth_masked.elevation_masked.odc.add_to(m, **options, name="depth_masked")

# Add layer control
folium.LayerControl().add_to(m)

m